# N043 · 二分答案与可行性判定

**目标：** 证明谓词单调后在答案空间搜索。

**先修：** N042, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最小可行/最大可行；上下界；整数向上取整；判定器成本。

**配套讲解来源：** [同名逐章意见](../../comment/043_binary_search_on_answer.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的问题叫"二分答案"。它指的是：题目不直接给你一个排好序的数组让你在里面找数，而是要你"求一个最小的、能满足条件的数"。比如两道经典题：

1. 猴子吃香蕉（Koko Eating Bananas）：有若干堆香蕉 `piles`，警卫会在 `h` 小时后回来，猴子每小时挑一堆、以每小时 `speed` 根的速度吃。问猴子**每小时至少吃几根**，才能在 `h` 小时内把香蕉全吃完。
2. 送货船（Capacity To Ship Packages）：货物按顺序 `weights` 逐日装船，每天运的重量不能超过船的容量 `capacity`，问船的**最小容量**是多少，才能在 `days` 天内运完。

这类问题的共同点是：直接"求最优值"很难，但反过来"给定一个候选答案，判断它行不行"很容易。所以我们的策略是：把"求最优"变成"判断可行性"，再在候选答案的空间上做二分查找。

给一个具体到数字的小例子（吃香蕉）：

- 输入：`piles = [3, 6, 7, 11]`，`h = 8`。
- 输出：`4`。
- 为什么是 4：如果每小时吃 4 根，吃各堆分别需要 `ceil(3/4)=1`、`ceil(6/4)=2`、`ceil(7/4)=2`、`ceil(11/4)=3` 小时，总计 `1+2+2+3 = 8` 小时，恰好不超过 8；而速度取 3 时需要 `1+2+3+4 = 10` 小时，超过 8，不够。所以 4 就是"最小可行速度"。

## 2. 基础知识：先读懂这些词

- **可行性判定（谓词 feasible）**：写一个函数，输入一个候选答案，输出"行/不行"（True/False）。它是把"求最优"翻译成"是/否判断"的桥梁。
- **单调性**：这是二分答案能成立的前提。它说的是"答案变大，判定只会从不行变行（或反之），不会来回抖"。比如吃香蕉：速度越快，总小时数只减不增；所以"总小时 ≤ h"这个条件一旦在某个速度下成立，比它更快的速度全都成立。没有单调性就不能二分（本章练习 2 就是让你构造反例体会这一点）。
- **最小可行 / 最大可行**：可行答案是一整片连续区间（比如速度 4 到 11 都行），我们要的是这片区间里最靠边的那个（最小的那个，即"第一个 True"）。
- **上下界（lo 与 hi）**：二分要在候选答案的范围内做。下界 `lo` 取"再小也不可能"的值（速度至少为 1；船容量至少是最重的那件货），上界 `hi` 取"一定可行"的值（速度最多是最大堆的大小，一次吃完；船容量最多是全部货物总和，一天运完）。**上界必须可行**，否则区间里根本没有答案，`first_true` 会直接报错。
- **整数向上取整 `(p + speed - 1) // speed`**：这是用整数运算实现 `ceil(p/speed)` 的技巧。一堆有 7 根、每小时吃 4 根，需要 `ceil(7/4)=2` 小时而不是 1.75 小时，因为最后一小时哪怕只剩 3 根，也得占用一整个小时。用 `(7+4-1)//4 = 10//4 = 2` 来算，全程整数，避免浮点误差——这正是 “思路推导”部分 强调的"整数天数用整除向上取整，避免浮点误差"。
- **判定器成本**：每判一次可行不可行要花 O(n) 的时间（要把每堆/每件货都扫一遍）。这个成本要算进总复杂度里。

## 3. 思路推导：从小例子开始

我们以吃香蕉 `piles=[3,6,7,11]`、`h=8` 为例，一步步走。

- **Step 1：写出判定器。** 给定速度 `speed`，算总小时数：每一堆独立计算 `ceil(p/speed)` 再求和（“正确性与复杂度”部分 说"吃香蕉判定可分离求和"，指的就是各堆互不影响、可以分别算）。判断"总小时数 ≤ h"。
- **Step 2：确定上下界。** 速度最小是 1（`lo=1`），最大是最大堆 11（`hi=11`，速度超过 11 没有意义，一堆最多 11 根，一小时吃 11 根就能吃空任何一堆）。
- **Step 3：验证单调性。** 速度越快，每堆所需小时只会减少或不变，所以总小时数不增。也就是说，可行速度是"前一段全是 False，后一段全是 True"的形状。
- **Step 4：二分找"第一个 True"。** 每次取中点 `mid` 判定：可行就把 `hi` 收到 `mid`（答案可能是 mid 或更小），不可行就把 `lo` 推到 `mid+1`（mid 本身不行，答案一定更大）。区间每轮减半，最后 `lo == hi` 就是答案。

**手算演示（二分过程一轮一轮演出来）：** 判定总小时数时逐堆算：

- 初始 `lo=1, hi=11`。`mid=6`：总小时 `ceil(3/6)+ceil(6/6)+ceil(7/6)+ceil(11/6)=1+1+2+2=6 ≤ 8`，可行 → `hi=6`。
- `lo=1, hi=6`。`mid=3`：总小时 `1+2+3+4=10 > 8`，不可行 → `lo=4`。
- `lo=4, hi=6`。`mid=5`：总小时 `ceil(3/5)+ceil(6/5)+ceil(7/5)+ceil(11/5)=1+2+2+3=8 ≤ 8`，可行 → `hi=5`。
- `lo=4, hi=5`。`mid=4`：总小时 `1+2+2+3=8 ≤ 8`，可行 → `hi=4`。
- `lo == hi == 4`，返回 4。只判定了 4 次，而不是从 1 到 11 逐个试 11 次。

**对照 “运行示例”部分 的小实例表格：** notebook 里 `show_table(['速度','总小时','≤8可行'], ...)` 对速度 1 到 11 逐个列出了总小时和可行性，数据是：

| 速度 | 总小时 | ≤8 可行 |
|---|---|---|
| 1 | 27 | 否 |
| 2 | 15 | 否 |
| 3 | 10 | 否 |
| 4 | 8 | 是 |
| 5 | 8 | 是 |
| 6 | 6 | 是 |
| 7 | 5 | 是 |
| 8 | 5 | 是 |
| 9 | 5 | 是 |
| 10 | 5 | 是 |
| 11 | 4 | 是 |

你可以亲眼看到"可行"从速度 4 开始连成一片——这就是单调性，而答案就是第一个"是"所在的 4。

## 4. 核心代码：min_eating_speed

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def min_eating_speed(piles, h):
    if not piles or h < len(piles):
        raise ValueError('positive piles and enough hours required')
    return first_true(1, max(piles), lambda speed: sum(((p + speed - 1) // speed for p in piles)) <= h)
```

### 逐段读懂代码

“分段实现”部分 一共三个函数，我们逐个看。

### 1. `first_true` —— 通用的"找第一个可行值"二分模板

```python
def first_true(lo, hi, feasible):
    """Smallest feasible integer in inclusive [lo,hi]; hi must be feasible."""
    if lo > hi or not feasible(hi):
        raise ValueError('upper bound must be feasible')
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo
```

- 第 3 行是防御性检查：如果区间是空的（`lo>hi`），或者上界 `hi` 本身都不可行（说明整个区间可能没有可行答案），就抛异常。这相当于在合同里写明"调用我之前，你必须保证 hi 可行"。
- `while lo<hi`：只要区间里还有多个候选值就继续二分。
- `mid=(lo+hi)//2`：整数中点，且偏左（向下取整）。因为 `lo < hi` 时 `mid` 一定小于 `hi`，所以 `hi=mid` 会让区间严格变小，不会死循环。
- `if feasible(mid): hi=mid`：mid 可行，说明答案在"mid 或更小"里，把上界压到 mid（mid 不敢扔掉，它可能就是答案）。
- `else: lo=mid+1`：mid 不可行，mid 一定不是答案，可以放心跳过它。
- 循环结束时 `lo==hi`，这个位置就是"第一个可行值"，直接返回 `lo`。

- 顺带说说"hi must be feasible"这条契约为什么重要：如果上界本身就不可行，说明你框定的搜索范围里可能一个可行答案都没有，继续二分没有意义（最后会返回一个不可行的假答案）。所以模板宁可立刻抛异常，逼调用者把上界给足。本章两个调用方都给足了：吃香蕉的上界 `max(piles)` 一定可行（每小时吃完一堆，总小时数等于堆数，而调用方已保证 `h>=len(piles)`）；装船的上界 `sum(weights)` 一定可行（一天全装完，而调用方已保证 `days>=1`）。

### 2. `min_eating_speed` —— 猴子吃香蕉

```python
def min_eating_speed(piles, h):
    if not piles or h < len(piles):
        raise ValueError('positive piles and enough hours required')
    return first_true(1, max(piles), lambda speed: sum(((p + speed - 1) // speed for p in piles)) <= h)
```

- 第 2 行挡掉两类非法输入：香蕉堆列表为空（`not piles`）；或者小时数比堆数还少（`h<len(piles)`，因为一小时只能吃一堆，堆数比小时多就永远吃不完）。这两种情况下问题无解，直接报错比返回一个错误答案好。
- 第 3 行把全部逻辑委托给 `first_true`：搜索范围 `[1, max(piles)]`（理由见第三节 Step 2），判定器是一个 lambda：对每堆 `p` 算 `(p+speed-1)//speed`（即 `ceil(p/speed)`），求和后与 `h` 比较。

### 3. `ship_within_days` —— 最小船容量

```python
def ship_within_days(weights, days):
    if not weights or days < 1:
        raise ValueError('nonempty positive weights and positive days required')

    def feasible(capacity):
        used = 1
        load = 0
        for w in weights:
            if load + w > capacity:
                used += 1
                load = 0
            load += w
        return used <= days
    return first_true(max(weights), sum(weights), feasible)
```

- 第 2 行挡住"货物为空"和"天数小于 1"这两类无意义输入。
- `feasible(capacity)` 是判定器：给定容量，模拟装货过程。`used` 是已用天数（从 1 开始，因为一开始就在第一天），`load` 是当前这天已装的重量。遍历每件货 `w`：如果当前天装不下它（`load+w>capacity`），就开新的一天（`used+=1`，并把 `load` 清零），然后把它装上（`load+=w`）。最后看总天数 `used <= days` 是否成立。
- 最后一行搜索范围是 `[max(weights), sum(weights)]`：容量小于最重货物时那件货永远上不了船（下界）；容量等于总重时一天全运完，一定可行（上界）。
- **再手算一遍装船**：`weights=[1,2,3,4]`、`days=2`，试 `capacity=7`：第一件 1 装上（load=1），第二件 2 装上（load=3），第三件 3 装上（load=6），第四件 4 时 `6+4=10>7`，开新一天（used=2，load=0 再装上 4）。总天数 2 ≤ 2，可行。而 `capacity=5` 时：1、2 装上（load=3），3 装不下开新天（load=3），4 又装不下再开新天，总天数 3 > 2，不可行。所以最小容量是 6（可以自己验证 capacity=6 恰好两天能装完）。

## 5. 分段实现：按顺序运行

**本章接口：** `min_eating_speed(piles, h)`、`ship_within_days(weights, days)`、`first_true(lo, hi, feasible)`

### first_true

In [1]:
def first_true(lo, hi, feasible):
    """Smallest feasible integer in inclusive [lo,hi]; hi must be feasible."""
    if lo > hi or not feasible(hi):
        raise ValueError('upper bound must be feasible')
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

### min_eating_speed

In [2]:
def min_eating_speed(piles, h):
    if not piles or h < len(piles):
        raise ValueError('positive piles and enough hours required')
    return first_true(1, max(piles), lambda speed: sum(((p + speed - 1) // speed for p in piles)) <= h)

### ship_within_days

In [3]:
def ship_within_days(weights, days):
    if not weights or days < 1:
        raise ValueError('nonempty positive weights and positive days required')

    def feasible(capacity):
        used = 1
        load = 0
        for w in weights:
            if load + w > capacity:
                used += 1
                load = 0
            load += w
        return used <= days
    return first_true(max(weights), sum(weights), feasible)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

p=[3,6,7,11]
show_table(['速度','总小时','≤8可行'],[(v,sum((x+v-1)//v for x in p),sum((x+v-1)//v for x in p)<=8) for v in range(1,12)])

速度,总小时,≤8可行
1,27,False
2,15,False
3,10,False
4,8,True
5,8,True
6,6,True
7,5,True
8,5,True
9,5,True
10,5,True


## 7. 正确性、适用条件与复杂度

吃香蕉判定可分离求和。装船判定中每一天尽量装到容量上限，不会让以后剩余前缀更多；对天数做“领先”归纳即可证明这是最少天数。单调可行集合的首个元素就是最优解。

**代价：** 每次判定 O(n)，二分次数 O(log U)，总 O(n log U)、O(1)辅助空间；U为答案搜索跨度。

### 展开理解

**吃香蕉为什么对？** 每一堆需要的小时数只取决于这一堆本身和速度，堆与堆之间互不干扰，所以总小时数就是把每堆的小时数加起来，这个判定器算的就是真实的总耗时，没有多算也没有漏算。

**装船为什么对？** 判定器用的是贪心策略：每一天都尽量往死里装，装不下才换下一天。你可能会问：万一"今天少装一点，明天更好安排"呢？不会的。你想，货物必须按顺序装，今天装得越多，留给后面的货物只会更少或一样，绝不可能更多，所以"今天使劲装"绝不会让后面的天数变多。反过来说，如果连"使劲装"都需要超过 `days` 天，那么任何别的装法也一定超过 `days` 天——因为别的装法每天装得不会更满。所以这个判定器给出的"行/不行"是准确的。

**二分为什么收敛到最优？** 因为可行性是单调的：不可行的全在左边，可行的全在右边。我们的二分每一步都在砍掉"确定不含答案"的那半边（mid 不可行就砍左半，mid 可行就砍右半但保留 mid）。最后 `lo==hi` 时，剩下那个数一定是"第一个可行"的，也就是最小的可行答案——这正是题目要的最优值。

**复杂度是多少？** 每判定一次要扫一遍所有堆（或所有货物），是 O(n)；二分每轮把区间砍半，区间跨度记作 U，总共只要判定约 log₂U 次。合起来是 O(n log U)，额外空间只有几个变量，是 O(1)。举个具体的数：假设有 10⁵ 堆香蕉、最大堆是 10⁹ 根，log₂(10⁹) 约等于 30，总共大约 3×10⁶ 次基本操作——也就是三百万次，普通电脑一瞬间就算完。对比一下：如果从速度 1 开始一个个往上试，最坏要试十亿次，每次还要扫十万个堆，那是十万亿级别的操作，根本算不完。这就是二分答案的价值。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `assert min_eating_speed([3,6,7,11],8)==4`：这是官方样例，属于**正常值测试**。我们在第三节已经手算过答案是 4，这条测试确认实现与手算一致。
- `assert ship_within_days([1,2,3,4],1)==10`：**极端边界——天数最少**。只有 1 天，所有货必须一天装完，容量只能等于总重 1+2+3+4=10。这条测试验证"上界确实可行、且恰好取在上界"。
- `assert ship_within_days([1,2,3,4],4)==4`：**另一个极端——天数最多**。4 天运 4 件货，每天一件，容量只要装下最重的那件（4）即可。这条测试验证"下界的选取是对的"（容量不能小于最大单件）。
- 接下来的双重循环是**暴力对拍**：`product([1,2,3],repeat=4)` 枚举所有由 1/2/3 组成的四堆组合（共 81 种），`h` 取 4 到 8。参考解 `ref` 是最朴素的写法——速度从 1 逐个往上试，找到第一个总小时 ≤ h 的速度。这保证我们的二分结果和暴力枚举在小规模数据上完全一致，这是抓 bug 最有效的手段。
- 最后一段循环 `for boundary in range(21): assert first_true(0,20,lambda x:x>=boundary)==boundary`：单独测 `first_true` 模板本身。谓词 `x>=boundary` 的"第一个 True"恰好是 `boundary`，枚举 boundary 从 0 到 20，覆盖了"全区间都可行（boundary=0）"到"只有上界可行（boundary=20）"的所有位置。这验证模板在边界放置上没有差一错误。
- `print('N043: 所有本章断言通过')` 只是打一行确认信息。

In [5]:
assert min_eating_speed([3, 6, 7, 11], 8) == 4

assert ship_within_days([1, 2, 3, 4], 1) == 10

assert ship_within_days([1, 2, 3, 4], 4) == 4

from itertools import product

for a in product([1, 2, 3], repeat=4):
    for h in range(4, 9):
        ref = next((v for v in range(1, max(a) + 1) if sum(((x + v - 1) // v for x in a)) <= h))
        assert min_eating_speed(a, h) == ref

for boundary in range(21):
    assert first_true(0, 20, lambda x: x >= boundary) == boundary

print('N043: 所有本章断言通过')

N043: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 为运力判定给出贪心装载论证。

**练习 2：** 构造不可单调谓词说明不能二分。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（为运力判定给出贪心装载论证）**：方向是把第四节里那段"今天装得越多，留给后面的只会更少"的直觉写成严谨论证。你可以先写清楚输入输出和边界（容量至少是 max(weights)，至多是 sum(weights)），然后拿 `weights=[1,2,3,4]、days=2` 手算一遍 capacity=5（不可行）和 capacity=6（可行）作对比示例，最后用本章的 `ship_within_days` 或自己写的逐容量枚举暴力解验证结论。关键是说清"交换论证"：任何一种装法如果某天没装满而提前开新天，都可以把后面的货挪一些过来，天数不会增加。
- **练习 2（构造不可单调谓词）**：方向是找一个"判定结果会来回变化"的谓词，比如"速度 v 恰好需要 h 小时"（总小时先随速度递减，等于 h 只在某一小段成立，之前之后都不成立），这样的 True 区间不是"左边全 False 右边全 True"的形状。你要用一个具体的小例子（比如 `piles=[3,6,7,11]`、谓词"总小时恰好等于 8"）列出速度 1 到 11 的判定表，展示 True 只在 4 和 5 出现，两边都是 False，然后说明：对这种谓词用 `first_true` 二分会跳过答案或返回错误位置。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def first_true(lo, hi, feasible):
    """Smallest feasible integer in inclusive [lo,hi]; hi must be feasible."""
    if lo > hi or not feasible(hi):
        raise ValueError('upper bound must be feasible')
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

def min_eating_speed(piles, h):
    if not piles or h < len(piles):
        raise ValueError('positive piles and enough hours required')
    return first_true(1, max(piles), lambda speed: sum(((p + speed - 1) // speed for p in piles)) <= h)

def ship_within_days(weights, days):
    if not weights or days < 1:
        raise ValueError('nonempty positive weights and positive days required')

    def feasible(capacity):
        used = 1
        load = 0
        for w in weights:
            if load + w > capacity:
                used += 1
                load = 0
            load += w
        return used <= days
    return first_true(max(weights), sum(weights), feasible)

# 示例与回归测试
assert min_eating_speed([3, 6, 7, 11], 8) == 4

assert ship_within_days([1, 2, 3, 4], 1) == 10

assert ship_within_days([1, 2, 3, 4], 4) == 4

from itertools import product

for a in product([1, 2, 3], repeat=4):
    for h in range(4, 9):
        ref = next((v for v in range(1, max(a) + 1) if sum(((x + v - 1) // v for x in a)) <= h))
        assert min_eating_speed(a, h) == ref

for boundary in range(21):
    assert first_true(0, 20, lambda x: x >= boundary) == boundary

print('N043: 所有本章断言通过')

N043: 所有本章断言通过


## 11. 代表题与迁移

- **875. Koko Eating Bananas（爱吃香蕉的猴子）**：就是本章 `min_eating_speed` 的原型题，练的是"最小可行速度 + 逐堆向上取整求和判定器"。
- **1011. Capacity To Ship Packages Within D Days（D 天内送达包裹的能力）**：就是本章 `ship_within_days` 的原型题，练的是"最小可行容量 + 顺序贪心装货判定器"。
- **410. Split Array Largest Sum（分割数组的最大值）**：把数组分成 k 段、最小化"最大段和"。它同样是二分答案：判定器变成"给定段和上限，贪心从头切段，看段数是否 ≤ k"，是本章两个判定器的直接迁移。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。